# Data Loading

In [1]:
import pandas as pd
data = pd.read_csv('C:/Users/user/Downloads/COD_TASK_1_Movie Genre Classification/Genre Classification Dataset/train_data.txt',sep=':::',
                   engine='python',header=None,names=['Id','Title','Genre','Plot'])
data.head()

,Id,Title,Genre,Plot
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...


# importing Libraries

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.utils import shuffle
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

# Null values checking

In [3]:
data.isnull().sum()

Id       0
Title    0
Genre    0
Plot     0
dtype: int64

# feature combining 

In [4]:
data['v1'] = data['Plot']
data['v2'] = data['Title'] + " " + data["Plot"]
data["v3"] = data["Title"] + " " + data["Title"] + " " +data["Plot"]

# downloading stopword and wordnet

In [5]:
nltk.download("stopwords")
nltk.download("wordnet")

stop_words  = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Text Preprocessing-1

In [6]:
def clean_text(text):
    text = str(text)
    text = re.sub('[^a-zA-Z]',' ',text)
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

In [7]:

data['v3'] = data['v3'].apply(clean_text)

# X and y define

In [8]:
X = data["v3"]
y = data["Genre"].astype(str).str.strip()

In [9]:
tfidf = TfidfVectorizer(max_features=20000,ngram_range=(1,3),min_df=3,max_df=0.85,sublinear_tf=True)

In [10]:
X, y = shuffle(X, y, random_state=42)

In [11]:
X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
X_train = tfidf.fit_transform(X_train_text)

In [13]:
# Transform test data
X_test = tfidf.transform(X_test_text)

# cross validation for every model and checking the score to see the model performance

In [14]:
model1 = Pipeline([
    ('tfidf', tfidf),
    ('clf', MultinomialNB())
])

In [15]:
scores1 = cross_val_score(model1, X, y, cv=3, n_jobs=-1)
scores1.mean()

np.float64(0.4893200956873609)

In [16]:
model2 = Pipeline([
    ('tfidf', tfidf),
    ('clf', LogisticRegression(max_iter=300,C=2,n_jobs=-1))
])
scores2 = cross_val_score(model2, X, y, cv=3, n_jobs=-1)
scores2.mean()

np.float64(0.5953996776131842)

In [17]:
for c in [0.1, 0.5, 1, 2, 5]:
    model3 = Pipeline([
        ('tfidf', tfidf),
        ('clf', LinearSVC(C=c))
    ])
    scores3 = cross_val_score(model3, X, y, cv=3, n_jobs=-1)
    print(f"C={c} -> {scores3.mean()}")

C=0.1 -> 0.5916921859554406
C=0.5 -> 0.5922086590767629
C=1 -> 0.5794444103945019
C=2 -> 0.5605747569468966
C=5 -> 0.5362821428701784


In [18]:
model = Pipeline([
    ('tfidf', tfidf),
    ('clf', SGDClassifier(loss='hinge', max_iter=1000))
])

scores = cross_val_score(model, X, y, cv=3, n_jobs=-1)
print("Cross-validation score:", scores.mean())


Cross-validation score: 0.5833548442008134


In [19]:
final_model = Pipeline([
    ('tfidf', tfidf),
    ('clf', LogisticRegression(C=3, max_iter=500,n_jobs=-1))
])
scores5 = cross_val_score(final_model, X, y, cv=3, n_jobs=-1)
scores5.mean()

np.float64(0.5949016553555125)

# Logistic Regression model selected

In [24]:
model2.fit(X_train_text,y_train)

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [25]:
y_pred = model2.predict(X_test_text)

In [26]:

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.595683851332657
